In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO)
import json

print("✓ Setup complete")

✓ Setup complete


In [2]:
from backend.extraction.value_normaliser import (
    normalise_currency, 
    normalise_date, 
    is_date_valid
)

# Test currency normalisation
test_cases = [
    ("Rs. 8,20,00,000",     82000000.0),
    ("8.2 crore",            82000000.0),
    ("820 lakh",             82000000.0),
    ("INR 82000000",         82000000.0),
    ("8.2 Cr",               82000000.0),
    ("Rs. 2,10,00,000",     21000000.0),
    ("2.1 crore",            21000000.0),
    ("5 Crore",              50000000.0),
    ("6,75,00,000",          67500000.0),
    ("Six Crore Twenty Lakhs", None),    # text format — may return None
]

print("Currency Normalisation Tests:")
print("-" * 55)
all_passed = True
for input_val, expected in test_cases:
    result = normalise_currency(input_val)
    status = "✓" if result == expected else "⚠"
    if result != expected:
        all_passed = False
    print(f"{status} '{input_val:30}' → {result} (expected {expected})")

print()

# Test date normalisation
date_tests = [
    "09/03/2027",
    "19/04/2025",
    "11/08/2027",
    "04/06/2028",
]
print("Date Validity Tests:")
print("-" * 40)
for d in date_tests:
    valid = is_date_valid(d)
    print(f"  {d} → {'VALID ✓' if valid else 'EXPIRED ✗'}")

print()
if all_passed:
    print("✓ All currency normalisation tests passed")
else:
    print("⚠ Some tests need attention — check value_normaliser.py")

Currency Normalisation Tests:
-------------------------------------------------------
✓ 'Rs. 8,20,00,000               ' → 82000000.0 (expected 82000000.0)
✓ '8.2 crore                     ' → 82000000.0 (expected 82000000.0)
✓ '820 lakh                      ' → 82000000.0 (expected 82000000.0)
✓ 'INR 82000000                  ' → 82000000.0 (expected 82000000.0)
✓ '8.2 Cr                        ' → 82000000.0 (expected 82000000.0)
✓ 'Rs. 2,10,00,000               ' → 21000000.0 (expected 21000000.0)
✓ '2.1 crore                     ' → 21000000.0 (expected 21000000.0)
✓ '5 Crore                       ' → 50000000.0 (expected 50000000.0)
✓ '6,75,00,000                   ' → 67500000.0 (expected 67500000.0)
✓ 'Six Crore Twenty Lakhs        ' → None (expected None)

Date Validity Tests:
----------------------------------------
  09/03/2027 → VALID ✓
  19/04/2025 → EXPIRED ✗
  11/08/2027 → VALID ✓
  04/06/2028 → VALID ✓

✓ All currency normalisation tests passed


In [3]:
from backend.extraction.ner_extractor import extract_entities

# Test on Bidder A balance sheet text
bidder_a_text = """
CHARTERED ACCOUNTANT CERTIFICATE
M/s Sharma Construction Pvt Ltd, GSTIN: 07AABCS1234A1Z5
FY 2023-24: Annual Turnover Rs. 8,20,00,000
FY 2022-23: Annual Turnover Rs. 7,45,00,000
FY 2021-22: Annual Turnover Rs. 6,80,00,000
CA Rajesh Kumar, Membership No: 123456
Date: 15/05/2024
"""

entities = extract_entities(bidder_a_text)

print("Entities extracted from Bidder A balance sheet:")
print("-" * 45)
for entity_type, values in entities.items():
    if values:
        print(f"{entity_type:25}: {values}")

# Test on GST certificate text
gst_text = """
GSTIN: 07AABCS1234A1Z5
Legal Name: Sharma Construction Pvt Ltd
Registration Status: ACTIVE
Date of Registration: 10/08/2018
"""

gst_entities = extract_entities(gst_text)
print("\nEntities from GST certificate:")
print("-" * 45)
for entity_type, values in gst_entities.items():
    if values:
        print(f"{entity_type:25}: {values}")

# Validate GST number was found
assert any('07AABCS1234A1Z5' in str(v) 
           for v in gst_entities.values()), \
    "GST number not extracted!"

print("\n✓ NER extraction working correctly")

Entities extracted from Bidder A balance sheet:
---------------------------------------------
GST_NO                   : ['07AABCS1234A1Z5']
PAN_NO                   : ['AABCS1234A']

Entities from GST certificate:
---------------------------------------------
GST_NO                   : ['07AABCS1234A1Z5']
PAN_NO                   : ['AABCS1234A']

✓ NER extraction working correctly


In [4]:
# --- Imports ---
from backend.ingestion.pdf_extractor import extract_digital_pdf
from backend.ingestion.chunker import chunk_document
from backend.extraction.section_classifier import classify_sections


# --- Fallback classifier (safe for notebook use) ---
def fallback_classification(chunks):
    keywords = [
        "eligibility", "qualification", "criteria",
        "requirement", "eligible", "disqualification"
    ]

    results = []
    for chunk in chunks:
        section_name = chunk.get("section_name", "").lower()
        text = chunk.get("text", "").lower()

        is_eligibility = any(kw in section_name for kw in keywords) or \
                         any(kw in text for kw in keywords)

        results.append({
            **chunk,
            "is_eligibility_section": is_eligibility
        })

    return results


# --- Step 1: Extract PDF ---
try:
    pages = extract_digital_pdf("../data/mock/tender_crpf_construction.pdf")
    print(f"✓ Extracted {len(pages)} pages")
except Exception as e:
    print(f"❌ PDF extraction failed: {e}")
    raise


# --- Step 2: Chunk document ---
chunks = chunk_document(pages)
print(f"✓ Created {len(chunks)} chunks")


# --- Step 3: Classify sections ---
try:
    classified = classify_sections(chunks)

    # Validate output
    if not classified or not isinstance(classified, list):
        raise ValueError("Invalid classification output")

except Exception as e:
    print(f"⚠️ LLM classification failed: {e}")
    print("⚠️ Switching to fallback classifier...")
    classified = fallback_classification(chunks)


# --- Step 4: Display results ---
print("\nSection classification results:")
print("-" * 50)

for chunk in classified:
    flag = "✓ ELIGIBILITY" if chunk.get("is_eligibility_section") else "  general"
    print(f"{flag} | {chunk.get('section_name', 'UNKNOWN')}")


# --- Step 5: Summary ---
eligibility_sections = [
    c for c in classified if c.get("is_eligibility_section")
]

count = len(eligibility_sections)

print("\n" + "-" * 50)
print(f"Eligibility sections found: {count}")

if count == 0:
    print("⚠️ No eligibility sections found")
    print("👉 Possible reasons:")
    print("   - Groq model outdated")
    print("   - PDF structure unusual")
    print("   - Section names missing keywords")
else:
    print("✓ Section classification working")


# --- Optional: Inspect detected eligibility sections ---
print("\nDetected Eligibility Sections:")
print("-" * 50)

for sec in eligibility_sections:
    print(f"- {sec.get('section_name')}")

✓ Extracted 1 pages
✓ Created 1 chunks

Section classification results:
--------------------------------------------------
✓ ELIGIBILITY | Eligibility Criteria

--------------------------------------------------
Eligibility sections found: 1
✓ Section classification working

Detected Eligibility Sections:
--------------------------------------------------
- Eligibility Criteria


In [5]:
from backend.extraction.criterion_extractor import extract_criteria

# Use the eligibility section text from previous cell
eligibility_text = eligibility_sections[0]['text']
section_name = eligibility_sections[0]['section_name']

print("Calling Groq API to extract criteria...")
print("This may take 5-10 seconds...\n")

criteria = extract_criteria(eligibility_text, section_name)

print(f"Total criteria extracted: {len(criteria)}")
print("=" * 60)

for c in criteria:
    mandatory_str = "MANDATORY ⚠" if c.mandatory else "desirable  ○"
    print(f"\n{c.criterion_id} [{mandatory_str}] — {c.criterion_type.upper()}")
    print(f"  Text:      {c.text[:80]}")
    print(f"  Threshold: {c.threshold} {c.unit or ''}")
    print(f"  Operator:  {c.operator}")
    print(f"  Evidence:  {c.evidence_docs}")

Calling Groq API to extract criteria...
This may take 5-10 seconds...



INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:backend.extraction.criterion_extractor:================================================================================
INFO:backend.extraction.criterion_extractor:RAW GROQ RESPONSE:
INFO:backend.extraction.criterion_extractor:================================================================================
INFO:backend.extraction.criterion_extractor:There are 5 numbered criteria in the given text. Here they are:
1. FINANCIAL CAPACITY
2. TECHNICAL EXPERIENCE
3. GST REGISTRATION
4. QUALITY CERTIFICATION
5. DEFENCE SECTOR EXPERIENCE

Here are the criteria in the required format:

### CRITERION ###
ID: C1
TEXT: The bidder shall have a minimum annual turnover of Rs. 5 Crore (Rupees Five Crore only) in each of the last three financial years (FY 2021-22, FY 2022-23, FY 2023-24).
TYPE: financial
MANDATORY: true
THRESHOLD: 5
OPERATOR: >=
UNIT: crore
EVIDENCE: Audited Balance Sheet, CA Certificat


[DEBUG Block 1] Keys parsed: ['there are 5 numbered criteria in the given text. here they are', 'here are the criteria in the required format']
[DEBUG Block 1] Values: {'there are 5 numbered criteria in the given text. here they are': '', 'here are the criteria in the required format': ''}

[DEBUG Block 2] Keys parsed: ['id', 'text', 'type', 'mandatory', 'threshold', 'operator', 'unit', 'evidence']
[DEBUG Block 2] Values: {'id': 'C1', 'text': 'The bidder shall have a minimum annual turnover of Rs. 5 Crore (Rupees Five Crore only) in each of the last three financial years (FY 2021-22, FY 2022-23, FY 2023-24).', 'type': 'financial', 'mandatory': 'true', 'threshold': '5', 'operator': '>=', 'unit': 'crore', 'evidence': 'Audited Balance Sheet, CA Certificate'}

[DEBUG Block 3] Keys parsed: ['id', 'text', 'type', 'mandatory', 'threshold', 'operator', 'unit', 'evidence']
[DEBUG Block 3] Values: {'id': 'C2', 'text': 'The bidder must have successfully completed at least 3 (three) similar const

In [6]:
print("Validating mandatory flags...")

mandatory_criteria = [c for c in criteria if c.mandatory]
optional_criteria = [c for c in criteria if not c.mandatory]

print(f"Mandatory criteria: {len(mandatory_criteria)} (expected 4)")
print(f"Optional criteria:  {len(optional_criteria)} (expected 1)")

assert len(mandatory_criteria) == 4, \
    f"Expected 4 mandatory criteria, got {len(mandatory_criteria)}"
assert len(optional_criteria) == 1, \
    f"Expected 1 optional criterion, got {len(optional_criteria)}"

# Check financial criterion has threshold
financial = [c for c in criteria if c.criterion_type == 'financial']
assert len(financial) >= 1, "No financial criterion found"
assert financial[0].threshold is not None, "Financial criterion missing threshold"
assert financial[0].threshold > 0, "Financial threshold should be > 0"

print("\n✓ Mandatory/optional flags correct")
print("✓ Financial threshold present")
print("✓ Criterion extraction fully validated")

Validating mandatory flags...
Mandatory criteria: 4 (expected 4)
Optional criteria:  1 (expected 1)

✓ Mandatory/optional flags correct
✓ Financial threshold present
✓ Criterion extraction fully validated


In [9]:
import json
from pathlib import Path

# Save extracted criteria for use in Notebook 3
output_path = Path("../data/outputs/criteria/extracted_criteria.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

criteria_data = [c.model_dump() for c in criteria]
with open(output_path, 'w') as f:
    json.dump(criteria_data, f, indent=2, default=str)

print(f"✓ Criteria saved to {output_path}")
print(f"  Total: {len(criteria_data)} criteria saved")

✓ Criteria saved to ..\data\outputs\criteria\extracted_criteria.json
  Total: 5 criteria saved
